# Inheritance and Substitution


```{contents}
:local:
:depth: 2
```


```{index} Substitution Is a Client's Behavioral Expectation
```



```{index} Substitution Is a Client's Behavioral Expectation
```

## Substitution Is a Client's Behavioral Expectation



A derived discount policy may add fields and implementation details, but a caller using the base policy contract should still receive a result satisfying that contract. Inheritance is therefore more than reusing a convenient constructor. Ask whether the derived object can be used wherever the base behavior is promised without forcing the caller to know which subtype it received.

For a contract that accepts nonnegative subtotals and returns a discount between zero and the subtotal, a derived implementation that rejects every subtotal below 100 has strengthened the accepted-input requirement. A caller that was valid for the base contract can now fail. A derived implementation returning more than the subtotal has weakened the promised result. Both problems remain even when the code compiles.

Construction also follows a responsibility order. Base initialization establishes base state; derived initialization establishes additional state. Do not duplicate base validation in every client or assume that a base reference removes the derived object's fields. Reference type controls what the caller can name directly; the object still has its complete runtime identity.

Guided comparison: an Order has a collection of lines. Is it a subtype of a line? No: composition represents that has-a relationship. A special discount policy can plausibly be a policy if it preserves the policy contract. Draw the client call and state the promise being relied on before choosing inheritance merely to avoid a few repeated assignments.

For related explanation, see the further reading below.[^development-reading]


Chapter 11 composed an order from products and lines. Each object had a clear job without inheriting from a custom base class. Inheritance becomes useful when one type promises to behave as a more general type and client code can depend on that relationship.

Use discounts to examine that promise. A checkout needs a policy that accepts a nonnegative subtotal and returns an amount between zero and that subtotal. Several implementations can satisfy the same behavioral expectation. This section first uses a base class; the final section compares that design with an interface.


The supporting [OOP examples and variations](1204_oop_examples.ipynb) retains the original programming-paradigm discussion, separation of concerns, and Vehicle/Car illustration alongside these business models.

```{index} Is-a Is a Behavioral Promise
```

## Is-a Is a Behavioral Promise




```{index} Is-a Is a Behavioral Promise
```


A **base class** defines members that a **derived class** inherits. A derived object can be used through a base-type reference. The practical question is whether the derived type preserves the expectations of clients written for the base type.

An order has products and lines, so it uses composition. A percentage discount can be one kind of discount policy, so a small hierarchy may fit. Do not choose inheritance only to avoid copying a few lines of code. Reusing implementation without preserving behavior can create misleading types.

| Relationship | Suitable starting design |
| --- | --- |
| An order contains lines. | Composition. |
| A line refers to a product. | Composition and explicit sharing. |
| Several discount kinds support the same operation. | A base class or interface. |
| Two classes happen to print similar text. | A helper method may be enough. |

A class can inherit directly from one class. That does not mean a program must put all related business concepts in one hierarchy. Keep the relationship shallow enough to explain with concrete requirements.


```{index} Base and Derived Members
```

## Base and Derived Members




```{index} Base and Derived Members
```


The `:` syntax names the base class. `PercentageDiscount` inherits the public `Apply` operation and supplies an override. A private base-class field would still belong to the object, but derived code could not access it directly. Public and protected operations should expose only what the design needs; protected is not a reason to make all fields accessible.

`base.Apply` explicitly requests the base implementation. Here it performs the input validation before the derived method calculates a different valid result.


In [1]:
using System;

DiscountPolicy ordinary = new DiscountPolicy();
PercentageDiscount member = new PercentageDiscount(0.10m);
Console.WriteLine($"Ordinary: {ordinary.Apply(100m):F2}");
Console.WriteLine($"Member: {member.Apply(100m):F2}");

class DiscountPolicy
{
    public virtual decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        return subtotal;
    }
}

class PercentageDiscount : DiscountPolicy
{
    private readonly decimal rate;
    public PercentageDiscount(decimal rate)
    {
        if (rate < 0m || rate > 1m) throw new ArgumentOutOfRangeException(nameof(rate));
        this.rate = rate;
    }
    public override decimal Apply(decimal subtotal)
    {
        decimal validSubtotal = base.Apply(subtotal);
        return validSubtotal * (1m - rate);
    }
}


Ordinary: 100.00
Member: 90.00


The results are 100.00 and 90.00. The percentage constructor validates its rate before storing it. The inherited input contract still matters: a negative subtotal is rejected by both implementations. Private state and validated operations keep the hierarchy's rules understandable.

Constructors are not inherited as ordinary methods. Each constructor must establish the valid state of its own type. When a base constructor requires arguments, use an initializer such as `: base(name)` to supply them before the derived constructor body runs.


```{index} Construction and Shared State
```

## Construction and Shared State




```{index} Construction and Shared State
```


The next example makes constructor chaining visible. `Employee` owns a validated name; `SalesEmployee` adds a sales target. The derived constructor supplies the base name and then validates its own target. Neither constructor reads keyboard input or prints a user interface.


In [1]:
using System;

SalesEmployee employee = new SalesEmployee("Avery", 5000m);
Console.WriteLine($"{employee.Name}: target {employee.Target:F2}");



class Employee
{
    public string Name { get; }
    public Employee(string name)
    {
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        Name = name.Trim();
    }
}
class SalesEmployee : Employee
{
    public decimal Target { get; }
    public SalesEmployee(string name, decimal target) : base(name)
    {
        if (target < 0m) throw new ArgumentOutOfRangeException(nameof(target));
        Target = target;
    }
}


Avery: target 5000.00


Constructor chaining does not mean the derived constructor creates a second employee object. It initializes the base portion of the same object. Draw one object with its base and derived state, and distinguish that from an order containing separate line objects.

Deciding whether an employee hierarchy is appropriate depends on application requirements. If employees can hold several changing roles, composing role objects may express the situation better than assigning one permanent subtype. A useful example should not become a rule that every organizational chart needs inheritance.


```{index} Substitution Preserves the Contract
```

## Substitution Preserves the Contract




```{index} Substitution Preserves the Contract
```


A **substitution** is using a derived object where a base type is expected. A client that accepts `DiscountPolicy` should continue to receive a valid result for a valid input. A derived class that returns a negative amount or rejects inputs the base accepts breaks that expectation even if its code compiles.

Write down the behavior you need before designing the hierarchy:

- Accept every nonnegative decimal subtotal in the exercise's domain.
- Reject a negative subtotal.
- Return an amount from zero through the original subtotal.
- Avoid changing the caller's order or printing from the calculation.

These are behavioral obligations. The compiler checks type compatibility and method signatures; it does not prove business invariants. Boundary checks help expose violations. Chapter 21 later develops more formal contract reasoning.


```{index} Practice: Preserve the Discount Contract
```

## Practice: Preserve the Discount Contract




```{index} Practice: Preserve the Discount Contract
```


Implement a fixed reduction that never produces a negative amount. Validate the configured reduction and preserve the base input check.


In [ ]:
using System;

DiscountPolicy policy = new FixedDiscount(5m);
Console.WriteLine(policy.Apply(20m).ToString("F2"));
Console.WriteLine(policy.Apply(3m).ToString("F2"));

class DiscountPolicy
{
    public virtual decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        return subtotal;
    }
}

class FixedDiscount : DiscountPolicy
{
    private readonly decimal reduction;
    public FixedDiscount(decimal reduction)
    {
        if (reduction < 0m) throw new ArgumentOutOfRangeException(nameof(reduction));
        this.reduction = reduction;
    }
    public override decimal Apply(decimal subtotal)
    {
        decimal validSubtotal = base.Apply(subtotal);
        // Your code starts here.
        return validSubtotal; // TODO: apply the reduction with a zero floor.
        // Your code ends here.
    }
}


In [1]:
using System;

DiscountPolicy policy = new FixedDiscount(5m);
Console.WriteLine(policy.Apply(20m).ToString("F2"));
Console.WriteLine(policy.Apply(3m).ToString("F2"));

class DiscountPolicy
{
    public virtual decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        return subtotal;
    }
}

class FixedDiscount : DiscountPolicy
{
    private readonly decimal reduction;
    public FixedDiscount(decimal reduction)
    {
        if (reduction < 0m) throw new ArgumentOutOfRangeException(nameof(reduction));
        this.reduction = reduction;
    }
    public override decimal Apply(decimal subtotal)
    {
        decimal validSubtotal = base.Apply(subtotal);
        return Math.Max(0m, validSubtotal - reduction);
    }
}


15.00
0.00


```{rubric} Footnotes
```


[^1201_inheritance-1]: Inheritance supports substitution only when derived behavior respects the relevant base contract. The examples use ordinary single class inheritance and do not require advanced generic variance.


[^development-reading]: Further reading: [Microsoft, Methods in C#](https://learn.microsoft.com/en-us/dotnet/csharp/methods). The worked business scenarios and guided reasoning here are original teaching examples.
